<a href="https://colab.research.google.com/github/utsavmakavana455-ops/EcoMove-Mobility-sustainable-urban-mobility-investigation/blob/main/Ecomove_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install dash


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 75.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [dash]


In [9]:
# EcoMove Mobility - Interactive Dashboard (COM7021)
# Built with Plotly Dash, following the same pattern as the lecture example:


import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

DATA_FILE = "EcoMoveMobility.xlsx"

# ---------------------------------------------------------------- data
df = pd.read_excel("/content/[5442] EcoMoveMobility (1).xlsx")
df["Date"] = pd.to_datetime(df["Date"], unit="D", origin="1899-12-30")   # Excel serial -> date
df["Month"] = df["Date"].dt.to_period("M").dt.to_timestamp()

MIN_DATE, MAX_DATE = df["Date"].min(), df["Date"].max()
MODES = sorted(df["Transport_Mode"].unique())
CITIES = sorted(df["City"].unique())
WEATHER = sorted(df["Weather"].unique())
ROUTES = sorted(df["Route_Type"].unique())

# one fixed colour per mode so a mode keeps its colour in every chart
MODE_COLOURS = dict(zip(MODES, px.colors.qualitative.Safe))

# ---------------------------------------------------------------- app
app = Dash(__name__)

CARD = {"flex": "1", "minWidth": "150px", "padding": "10px", "margin": "5px",
        "border": "1px solid #dddddd", "borderRadius": "6px", "textAlign": "center",
        "backgroundColor": "#f7f9fb"}


def dropdown(label, element_id, options):
    """Multi-select dropdown; empty selection means 'all'."""
    return html.Div([
        html.Label(label),
        dcc.Dropdown(id=element_id, options=options, multi=True, placeholder="All"),
    ], style={"flex": "1", "minWidth": "180px", "margin": "5px"})


app.layout = html.Div([
    html.H1("EcoMove Sustainable Mobility Dashboard"),
    html.P("For senior managers: demand, reliability, finance, accessibility and sustainability, 2024-2025. "
           "Leave a filter empty to include everything."),

    # ---- filters
    html.Div([
        dropdown("City", "city-filter", CITIES),
        dropdown("Transport mode", "mode-filter", MODES),
        dropdown("Route type", "route-filter", ROUTES),
        dropdown("Weather", "weather-filter", WEATHER),
        dropdown("Event day", "event-filter", ["Yes", "No"]),
    ], style={"display": "flex", "flexWrap": "wrap"}),

    html.Div([
        html.Label("Date range"),
        dcc.DatePickerRange(id="date-filter", min_date_allowed=MIN_DATE, max_date_allowed=MAX_DATE,
                            start_date=MIN_DATE, end_date=MAX_DATE, display_format="DD MMM YYYY"),
    ], style={"margin": "5px"}),

    # ---- KPI cards (filled by the callback)
    html.Div(id="kpi-row", style={"display": "flex", "flexWrap": "wrap", "marginTop": "10px"}),

    # ---- charts: overview -> patterns -> detail
    html.H3("1. Overview"),
    html.Div([
        dcc.Graph(id="trend-chart", style={"flex": "2", "minWidth": "400px"}),
        dcc.Graph(id="city-chart", style={"flex": "1", "minWidth": "300px"}),
    ], style={"display": "flex", "flexWrap": "wrap"}),

    html.H3("2. Reliability and customer experience"),
    html.Div([
        dcc.Graph(id="weather-chart", style={"flex": "1", "minWidth": "350px"}),
        dcc.Graph(id="scatter-chart", style={"flex": "1", "minWidth": "350px"}),
    ], style={"display": "flex", "flexWrap": "wrap"}),

    html.H3("3. Finance, accessibility and sustainability by mode"),
    html.Div([
        dcc.Graph(id="finance-chart", style={"flex": "1", "minWidth": "300px"}),
        dcc.Graph(id="access-chart", style={"flex": "1", "minWidth": "300px"}),
        dcc.Graph(id="co2-chart", style={"flex": "1", "minWidth": "300px"}),
    ], style={"display": "flex", "flexWrap": "wrap"}),
], style={"fontFamily": "Arial, sans-serif", "margin": "20px"})


# ---------------------------------------------------------------- helpers
def apply_filters(city, mode, route, weather, event, start, end):
    """Return the rows that match every filter that has been chosen."""
    data = df[(df["Date"] >= pd.to_datetime(start)) & (df["Date"] <= pd.to_datetime(end))]
    for column, chosen in [("City", city), ("Transport_Mode", mode), ("Route_Type", route),
                           ("Weather", weather), ("Event_Day", event)]:
        if chosen:
            data = data[data[column].isin(chosen)]
    return data


def empty_figure(message="No data for the selected filters"):
    fig = px.scatter()
    fig.update_layout(title=message, xaxis={"visible": False}, yaxis={"visible": False})
    return fig


def kpi_card(label, value):
    return html.Div([html.H2(value, style={"margin": "0", "color": "#1f4e79"}),
                     html.Span(label)], style=CARD)


# ---------------------------------------------------------------- callback
@app.callback(
    [Output("kpi-row", "children"),
     Output("trend-chart", "figure"),
     Output("city-chart", "figure"),
     Output("weather-chart", "figure"),
     Output("scatter-chart", "figure"),
     Output("finance-chart", "figure"),
     Output("access-chart", "figure"),
     Output("co2-chart", "figure")],
    [Input("city-filter", "value"),
     Input("mode-filter", "value"),
     Input("route-filter", "value"),
     Input("weather-filter", "value"),
     Input("event-filter", "value"),
     Input("date-filter", "start_date"),
     Input("date-filter", "end_date")]
)
def update_dashboard(city, mode, route, weather, event, start, end):
    data = apply_filters(city, mode, route, weather, event, start, end)

    if data.empty:
        blank = empty_figure()
        return [html.P("No records match these filters.")] + [blank] * 7

    # ---- KPI cards
    pax = data["Passengers"].sum()
    kpis = [
        kpi_card("Records selected", f"{len(data):,}"),
        kpi_card("Passengers", f"{pax / 1e6:.2f}M"),
        kpi_card("Cost recovery", f"{data['Ticket_Revenue_EUR'].sum() / data['Operating_Cost_EUR'].sum() * 100:.1f}%"),
        kpi_card("Avg delay (min)", f"{data['Average_Delay_Minutes'].mean():.2f}"),
        kpi_card("Avg on-time", f"{data['On_Time_Percentage'].mean():.1f}%"),
        kpi_card("Avg rating", f"{data['Customer_Rating'].mean():.2f}"),
        kpi_card("CO2 saved (t)", f"{data['CO2_Saved_KG'].sum() / 1000:,.0f}"),
        kpi_card("Complaints / 1,000 pax", f"{data['Accessibility_Complaints'].sum() / pax * 1000:.2f}"),
    ]

    # small-sample warning: charts built on very few records are unstable
    if len(data) < 100:
        kpis.append(html.Div(f"Only {len(data)} records match these filters - treat the charts with caution.",
                             style={"width": "100%", "margin": "5px", "color": "#b00020", "fontWeight": "bold"}))

    # ---- 1. monthly passengers (line)
    monthly = data.groupby("Month", as_index=False)["Passengers"].sum()
    trend = px.line(monthly, x="Month", y="Passengers", markers=True,
                    title="Monthly passengers")

    # ---- 2. passengers by city (bar)
    city_data = (data.groupby("City", as_index=False)["Passengers"].sum()
                 .sort_values("Passengers"))
    city_fig = px.bar(city_data, x="Passengers", y="City", orientation="h",
                      title="Passengers by city")

    # ---- 3. delay by weather (box)
    weather_fig = px.box(data, x="Weather", y="Average_Delay_Minutes",
                         category_orders={"Weather": ["Clear", "Cloudy", "Windy", "Heatwave", "Rain", "Snow"]},
                         title="Delay by weather (minutes)")

    # ---- 4. delay v rating (scatter, sampled so the page stays fast)
    sample = data.sample(min(len(data), 1500), random_state=1)
    scatter = px.scatter(sample, x="Average_Delay_Minutes", y="Customer_Rating",
                         color="Transport_Mode", color_discrete_map=MODE_COLOURS, opacity=0.5,
                         hover_data=["City", "Weather"], title="Delay v customer rating")

    # ---- 5. cost recovery by mode (bar with break-even line)
    fin = data.groupby("Transport_Mode").agg(Revenue=("Ticket_Revenue_EUR", "sum"),
                                              Cost=("Operating_Cost_EUR", "sum")).reset_index()
    fin["Cost recovery (%)"] = fin["Revenue"] / fin["Cost"] * 100
    finance = px.bar(fin, x="Transport_Mode", y="Cost recovery (%)", color="Transport_Mode",
                     color_discrete_map=MODE_COLOURS, title="Cost recovery by mode (100% = break-even)")
    finance.add_hline(y=100, line_dash="dash")
    finance.update_layout(showlegend=False)

    # ---- 6. complaints per 1,000 passengers by mode
    acc = data.groupby("Transport_Mode").agg(Complaints=("Accessibility_Complaints", "sum"),
                                              Passengers=("Passengers", "sum")).reset_index()
    acc["Complaints per 1,000 passengers"] = acc["Complaints"] / acc["Passengers"] * 1000
    access = px.bar(acc, x="Transport_Mode", y="Complaints per 1,000 passengers", color="Transport_Mode",
                    color_discrete_map=MODE_COLOURS, title="Accessibility complaints per 1,000 passengers")
    access.update_layout(showlegend=False)

    # ---- 7. CO2 saved per passenger by mode
    env = data.groupby("Transport_Mode").agg(CO2=("CO2_Saved_KG", "sum"),
                                              Passengers=("Passengers", "sum")).reset_index()
    env["CO2 saved per passenger (kg)"] = env["CO2"] / env["Passengers"]
    co2 = px.bar(env, x="Transport_Mode", y="CO2 saved per passenger (kg)", color="Transport_Mode",
                 color_discrete_map=MODE_COLOURS, title="CO2 saved per passenger")
    co2.update_layout(showlegend=False)

    for fig in (trend, city_fig, weather_fig, scatter, finance, access, co2):
        fig.update_layout(margin={"l": 40, "r": 20, "t": 50, "b": 40}, template="plotly_white")

    return kpis, trend, city_fig, weather_fig, scatter, finance, access, co2


if __name__ == "__main__":
    app.run(port=8050, debug=False)

<IPython.core.display.Javascript object>